# Kimodo Raw Output Video Visualization

This notebook loads the raw `motion.npz` generated by Kimodo, draws the skeleton frame by frame, and saves a `.mp4` displayed inline.

## Prerequisites

- A `motion.npz` file (standard Kimodo output)
- Environment with `matplotlib` + `imageio-ffmpeg`

Expected fields in the raw file:
- `posed_joints`: shape `(T, J, 3)`
- optional: `root_positions`, `global_rot_mats`, `local_rot_mats`, `foot_contacts`

In [1]:
# Install dependencies if needed
%pip install -q numpy matplotlib imageio imageio-ffmpeg ipython

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import sys

import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Video, display

def import_kimodo_skeleton_definitions():
    kimodo_repo = Path("kimodo").resolve()
    kimodo_repo_str = str(kimodo_repo)

    # Ensure the correct package is resolved before namespace paths.
    if kimodo_repo_str not in sys.path:
        sys.path.insert(0, kimodo_repo_str)

    # Clear cached module to avoid collision with kimodo/assets at repo root.
    for module_name in list(sys.modules):
        if module_name == "kimodo" or module_name.startswith("kimodo."):
            sys.modules.pop(module_name, None)

    from kimodo.skeleton.definitions import SOMASkeleton30, SOMASkeleton77, SMPLXSkeleton22
    return SOMASkeleton30, SOMASkeleton77, SMPLXSkeleton22

try:
    from kimodo.skeleton.definitions import SOMASkeleton30, SOMASkeleton77, SMPLXSkeleton22
except Exception as exc:
    SOMASkeleton30, SOMASkeleton77, SMPLXSkeleton22 = import_kimodo_skeleton_definitions()
    print(f"Kimodo import adjusted via sys.path ({type(exc).__name__}).")

plt.style.use("seaborn-v0_8-whitegrid")
np.set_printoptions(suppress=True, precision=4)

/home/henriquesouza/miniconda3/envs/kimodo/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
DOMAIN = "10ms"
USER = "u02"
TAG = "tag11"
WINDOW = "w000"

# ===== CONFIGURATION =====
RUN_DIR = Path(
    f"output/robot_emotions_kimodo_generated_single/robot_emotions_{DOMAIN}_{USER}_{TAG}__{WINDOW}"
)
RAW_NPZ_PATH = RUN_DIR / "motion.npz"
OUT_VIDEO = RUN_DIR / "motion_raw_preview.mp4"
FPS = 20
FIG_SIZE = (7, 7)
LINE_WIDTH = 2.0
JOINT_SIZE = 18
MAX_FRAMES = None  # e.g.: 120 for quick test

if not RAW_NPZ_PATH.exists():
    candidates = sorted(Path("output").glob("robot_emotions_kimodo_generated_single*/**/motion.npz"))
    if not candidates:
        raise FileNotFoundError(
            "No motion.npz found in output/robot_emotions_kimodo_generated_single*/**/."
        )
    RAW_NPZ_PATH = candidates[0]
    RUN_DIR = RAW_NPZ_PATH.parent
    OUT_VIDEO = RUN_DIR / "motion_raw_preview.mp4"

print(f"RAW_NPZ_PATH: {RAW_NPZ_PATH}")
print(f"OUT_VIDEO: {OUT_VIDEO}")

FileNotFoundError: No motion.npz found in output/robot_emotions_kimodo_generated_single*/**/.

In [ ]:
# ===== LOADING =====
data = np.load(RAW_NPZ_PATH, allow_pickle=True)

print("Keys found in file:")
for key in data.files:
    value = data[key]
    shape = getattr(value, "shape", None)
    dtype = getattr(value, "dtype", type(value))
    print(f"  - {key}: {shape}, dtype={dtype}")

if "posed_joints" not in data:
    raise KeyError("Raw motion.npz must contain the 'posed_joints' key.")

posed_joints = np.asarray(data["posed_joints"], dtype=np.float32)
if posed_joints.ndim != 3 or posed_joints.shape[-1] != 3:
    raise ValueError(
        f"posed_joints must have shape (T, J, 3); got {posed_joints.shape}."
    )

if MAX_FRAMES is not None:
    posed_joints = posed_joints[:MAX_FRAMES]

T, J, _ = posed_joints.shape
print(f"\nNumber of frames: {T}")
print(f"Number of joints: {J}")

In [ ]:
# ===== SKELETON TOPOLOGY =====
def _parents_from_definition(skeleton_cls):
    names = [name for name, _ in skeleton_cls.bone_order_names_with_parents]
    parent_by_name = dict(skeleton_cls.bone_order_names_with_parents)
    name_to_index = {name: idx for idx, name in enumerate(names)}
    parents = np.array(
        [
            -1 if parent_by_name[name] is None else name_to_index[parent_by_name[name]]
            for name in names
        ],
        dtype=np.int64,
    )
    return names, parents


def infer_topology(num_joints):
    candidates = [SOMASkeleton77, SOMASkeleton30, SMPLXSkeleton22]
    for skeleton_cls in candidates:
        names, parents = _parents_from_definition(skeleton_cls)
        if len(names) == num_joints:
            return names, parents, skeleton_cls.__name__

    names = [f"joint_{i:02d}" for i in range(num_joints)]
    parents = np.arange(num_joints, dtype=np.int64) - 1
    parents[0] = -1
    return names, parents, "fallback_chain"


joint_names, joint_parents, topology_name = infer_topology(J)
edges = [
    (parent_idx, child_idx)
    for child_idx, parent_idx in enumerate(joint_parents.tolist())
    if parent_idx >= 0 and parent_idx < J
]

print(f"Topology used: {topology_name}")
print(f"Total edges: {len(edges)}")

In [ ]:
# ===== RENDERING =====
def render_raw_kimodo_video(
    points,
    edges,
    out_path,
    fps=20,
    fig_size=(7, 7),
    line_width=2.0,
    joint_size=18,
    title_prefix="Kimodo raw",
):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    mins = points.reshape(-1, 3).min(axis=0)
    maxs = points.reshape(-1, 3).max(axis=0)
    center = (mins + maxs) / 2.0
    span = float(np.max(maxs - mins))
    span = max(span, 1e-3)
    half = 0.6 * span

    writer = imageio.get_writer(
        str(out_path),
        fps=fps,
        codec="libx264",
        macro_block_size=None,
    )

    fig = plt.figure(figsize=fig_size)
    ax = fig.add_subplot(111, projection="3d")

    try:
        for t in range(points.shape[0]):
            ax.cla()
            frame = points[t]

            ax.scatter(frame[:, 0], frame[:, 1], frame[:, 2], s=joint_size, c="#154360")
            for parent_idx, child_idx in edges:
                segment = frame[[parent_idx, child_idx]]
                ax.plot(
                    segment[:, 0],
                    segment[:, 1],
                    segment[:, 2],
                    c="#2E86C1",
                    linewidth=line_width,
                )

            ax.set_xlim(center[0] - half, center[0] + half)
            ax.set_ylim(center[1] - half, center[1] + half)
            ax.set_zlim(center[2] - half, center[2] + half)
            ax.set_xlabel("X")
            ax.set_ylabel("Y")
            ax.set_zlabel("Z")
            ax.set_title(f"{title_prefix} - frame {t + 1}/{points.shape[0]}")
            ax.view_init(elev=20, azim=-65)

            fig.canvas.draw()
            rgba = np.asarray(fig.canvas.buffer_rgba(), dtype=np.uint8)
            image = rgba[..., :3].copy()
            writer.append_data(image)

            if (t + 1) % 20 == 0 or t == points.shape[0] - 1:
                print(f"Rendered: {t + 1}/{points.shape[0]}")
    finally:
        writer.close()
        plt.close(fig)

    return out_path


rendered_path = render_raw_kimodo_video(
    points=posed_joints,
    edges=edges,
    out_path=OUT_VIDEO,
    fps=FPS,
    fig_size=FIG_SIZE,
    line_width=LINE_WIDTH,
    joint_size=JOINT_SIZE,
    title_prefix=topology_name,
)
print(f"Video saved to: {rendered_path}")

In [ ]:
# ===== INLINE DISPLAY =====
display(Video(str(rendered_path), embed=True))

## Debug Notes

- If `motion.npz` has a joint count other than 77/30/22, the notebook falls back to `fallback_chain` topology.
- For fast rendering, set `MAX_FRAMES` in the configuration cell.
- If the MP4 does not open, reinstall `imageio-ffmpeg` and restart the kernel.
- If you want an SMPL-X mesh instead of the raw skeleton, use the smplx notebook flow with `motion_amass.npz`.